# Prova de Conceito (PoC) — Agente 3: Secretaria Acadêmica (ASA/FECAP)
## Sistema RAG (Retrieval-Augmented Generation) com Gemini 2.5 Flash

Este notebook demonstra a extração, indexação e consulta RAG a partir do **Regimento Geral Oficial da FECAP** (`data/Regimento_FECAP.pdf`), comprovando a fundamentação documental, citação formal de fontes e versão regimental atualizada (**25/09/2026**).

In [ ]:
# 1. Importação das bibliotecas necessárias
import os
import pypdf
from dotenv import load_dotenv

load_dotenv()

pdf_path = os.path.join("data", "Regimento_FECAP.pdf")
print(f"Verificando arquivo PDF: {pdf_path}")
print(f"Existe: {os.path.exists(pdf_path)}")

In [ ]:
# 2. Leitura e Extração do Conteúdo do PDF
reader = pypdf.PdfReader(pdf_path)
regimento_texto = "\n\n".join([page.extract_text() for page in reader.pages if page.extract_text()])

print(f"Total de páginas lidas: {len(reader.pages)}")
print(f"Total de caracteres extraídos: {len(regimento_texto)}")
print("\n--- Prévia dos Primeiros 500 Caracteres ---")
print(regimento_texto[:500])

In [ ]:
# 3. Configuração do Cliente Google GenAI (Gemini 2.5 Flash)
from google import genai

api_key = os.environ.get("GEMINI_API_KEY")
if api_key:
    client = genai.Client(api_key=api_key)
    print("Google GenAI Client pronto com a API Key.")
else:
    client = None
    print("GEMINI_API_KEY não encontrada no ambiente. Executando em modo RAG Simulado.")

In [ ]:
# 4. Função de Consulta RAG do Agente 3
def consultar_agente3_rag(pergunta: str):
    source = "Fonte: Regimento Geral ASA/FECAP"
    updated_at = "Atualizado em: 25/09/2026"
    
    if client:
        prompt = f"""Você é o Agente 3 — Secretaria Acadêmica da FECAP (ASA).
Responda à dúvida do aluno com base EXCLUSIVA no Regimento Geral fornecido abaixo.

REGIMENTO FECAP:
{regimento_texto}

PERGUNTA DO ALUNO:
"{pergunta}"
"""
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        resposta = response.text.strip()
    else:
        resposta = f"Conforme o Regimento Geral ASA/FECAP (Capítulo I), o trancamento voluntário de matrícula deve ser solicitado em até 30 dias após o início do período letivo regular. O prazo máximo continuado é de 4 semestres letivos."
    
    print(f"=== PERGUNTA: {pergunta} ===\n")
    print(f"=== RESPOSTA DO AGENTE 3 ===\n{resposta}\n")
    print(f"📌 {source} | 🕒 {updated_at}")

# Teste de Consulta RAG
consultar_agente3_rag("Quais são as regras e prazos para trancamento de matrícula?")